Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
collection = client.create_collection(
    name="disease_collection",
    embedding_function=sentence_transformer_ef
)

In [5]:
import pandas as pd
import ast
import numpy as np
from sentence_transformers import SentenceTransformer
master_df = pd.read_csv("C:/Users/Kashish U Singh/master_disease_data.csv")
print(master_df.head())

          Disease                                           Symptoms  \
0  panic disorder  ['anxiety and nervousness', 'shortness of brea...   
1  panic disorder  ['shortness of breath', 'depressive or psychot...   
2  panic disorder  ['anxiety and nervousness', 'depression', 'sho...   
3  panic disorder  ['anxiety and nervousness', 'depressive or psy...   
4  panic disorder  ['anxiety and nervousness', 'depression', 'ins...   

                                         Description  \
0  Panic disorder is a mental health condition ma...   
1  Panic disorder is a mental health condition ma...   
2  Panic disorder is a mental health condition ma...   
3  Panic disorder is a mental health condition ma...   
4  Panic disorder is a mental health condition ma...   

                                          Medication  \
0  ['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benz...   
1  ['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benz...   
2  ['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benz...   
3  ['S

In [7]:
def to_list(val):
    if pd.isna(val) or val == "":
        return []
    if isinstance(val, list):
        return val
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
        return [str(parsed).strip()]
    except:
        return [str(val).strip()]

master_df["Symptoms"] = master_df["Symptoms"].apply(to_list)
master_df["Medication"] = master_df["Medication"].apply(to_list)
master_df["Precautions"] = master_df["Precautions"].apply(to_list)
master_df["Diet"] = master_df["Diet"].apply(to_list)
master_df["Workout"] = master_df["Workout"].apply(to_list)

In [ ]:
def make_retrieval_text(row):
    disease = row["Disease"]
    symptoms = ", ".join(row["Symptoms"]) if isinstance(row["Symptoms"], list) else ""
    
    retrieval_text = f"""
    Disease: {disease}
    Symptoms: {symptoms}
    """
    
    return retrieval_text.strip()

master_df["retrieval_text"] = master_df.apply(make_retrieval_text, axis=1)

In [9]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

In [11]:
collection = client.create_collection(
    name="disease1_collection",
    embedding_function=sentence_transformer_ef
)

In [13]:
def make_retrieval_text(row):
    disease = row["Disease"]
    symptoms = ", ".join(row["Symptoms"]) if isinstance(row["Symptoms"], list) else ""
    
    retrieval_text = f"""
    Disease: {disease}
    Symptoms: {symptoms}
    """
    
    return retrieval_text.strip()

master_df["retrieval_text"] = master_df.apply(make_retrieval_text, axis=1)

print(master_df[["Disease", "retrieval_text"]].head())

          Disease                                     retrieval_text
0  panic disorder  Disease: panic disorder\n    Symptoms: anxiety...
1  panic disorder  Disease: panic disorder\n    Symptoms: shortne...
2  panic disorder  Disease: panic disorder\n    Symptoms: anxiety...
3  panic disorder  Disease: panic disorder\n    Symptoms: anxiety...
4  panic disorder  Disease: panic disorder\n    Symptoms: anxiety...


In [16]:
master_df = master_df.drop_duplicates(subset=["Disease"])
print(len(master_df))

100


In [17]:
documents = master_df["retrieval_text"].tolist()
metadatas = [{"disease": d} for d in master_df["Disease"].tolist()]
ids = [str(i) for i in range(len(master_df))]

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("Data inserted into ChromaDB")

Data inserted into ChromaDB


In [18]:
def search_disease_chroma(user_query, top_k=10):
    results = collection.query(
        query_texts=[user_query],
        n_results=top_k
    )

    diseases = []
    for i in range(len(results["metadatas"][0])):
        diseases.append({
            "disease": results["metadatas"][0][i]["disease"],
            "distance": results["distances"][0][i]
        })

    return diseases

In [19]:
def build_user_query(symptom_list):
    return "Symptoms: " + ", ".join([s.strip().lower() for s in symptom_list])

In [20]:
def exact_symptom_match(user_symptoms):
    user_symptoms = set([s.strip().lower() for s in user_symptoms])
    results = []

    for _, row in master_df.iterrows():
        disease = row["Disease"]
        disease_symptoms = set([s.strip().lower() for s in row["Symptoms"]])

        overlap = len(user_symptoms.intersection(disease_symptoms))
        total_user = len(user_symptoms)
        match_ratio = overlap / total_user if total_user > 0 else 0

        results.append({
            "disease": disease,
            "overlap": overlap,
            "match_ratio": match_ratio
        })

    results = sorted(results, key=lambda x: (-x["overlap"], -x["match_ratio"]))
    return results

In [21]:
def hybrid_rank(user_symptoms, top_k_chroma=20):
    query = build_user_query(user_symptoms)

    chroma_results = search_disease_chroma(query, top_k=top_k_chroma)
    chroma_map = {r["disease"].lower(): r["distance"] for r in chroma_results}

    exact_results = exact_symptom_match(user_symptoms)

    combined = []
    for r in exact_results:
        disease = r["disease"]
        disease_key = disease.lower()

        chroma_distance = chroma_map.get(disease_key, 999.0)

        combined.append({
            "disease": disease,
            "overlap": r["overlap"],
            "match_ratio": r["match_ratio"],
            "chroma_distance": chroma_distance
        })

    combined = sorted(
        combined,
        key=lambda x: (-x["overlap"], -x["match_ratio"], x["chroma_distance"])
    )

    return combined

In [22]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "Kashish@2001"

driver = GraphDatabase.driver(uri, auth=(username, password))
print("Connected to Neo4j successfully.")

Connected to Neo4j successfully.


In [23]:
def get_disease_details(disease_name):
    query = """
    MATCH (d:Disease {name: $disease})
    OPTIONAL MATCH (s:Symptom)-[:INDICATES]->(d)
    OPTIONAL MATCH (d)-[:TREATED_BY]->(m:Medication)
    OPTIONAL MATCH (d)-[:NEEDS_PRECAUTION]->(p:Precaution)
    OPTIONAL MATCH (d)-[:RECOMMENDS_DIET]->(di:Diet)
    OPTIONAL MATCH (d)-[:SUGGESTS_WORKOUT]->(w:Workout)
    RETURN d.name AS disease,
           d.description AS description,
           collect(DISTINCT s.name) AS symptoms,
           collect(DISTINCT m.name) AS medications,
           collect(DISTINCT p.name) AS precautions,
           collect(DISTINCT di.name) AS diets,
           collect(DISTINCT w.name) AS workouts
    """
    with driver.session() as session:
        result = session.run(query, disease=disease_name.lower())
        return [record.data() for record in result]

In [24]:
import ast

def flatten_field(val):
    if isinstance(val, list):
        flat = []
        for item in val:
            if isinstance(item, str):
                try:
                    parsed = ast.literal_eval(item)
                    if isinstance(parsed, list):
                        flat.extend([str(x).strip() for x in parsed])
                    else:
                        flat.append(item.strip())
                except:
                    flat.append(item.strip())
            else:
                flat.append(str(item).strip())
        return flat
    return []

In [25]:
def clean_result_list(values):
    if not values:
        return []
    return flatten_field(values)

In [26]:
def generate_response(symptoms):
    ranked_results = hybrid_rank(symptoms)

    final_output = []

    for r in ranked_results[:3]:
        disease_name = r["disease"]
        details = get_disease_details(disease_name)

        if details:
            d = details[0]

            medications = clean_result_list(d["medications"])
            precautions = clean_result_list(d["precautions"])
            diets = clean_result_list(d["diets"])
            workouts = clean_result_list(d["workouts"])
            disease_symptoms = clean_result_list(d["symptoms"])

            output = f"""
Possible Condition: {d['disease']}
Symptom Match Count: {r['overlap']}
Match Ratio: {r['match_ratio']:.2f}

Description:
{d['description']}

Common Symptoms:
{', '.join(disease_symptoms[:6])}

Medications:
{', '.join(medications[:5])}

Precautions:
{', '.join(precautions[:5])}

Diet Suggestions:
{', '.join(diets[:5])}

Workouts:
{', '.join(workouts[:5])}
"""
            final_output.append(output.strip())

    return "\n\n----------------------\n\n".join(final_output)

In [27]:
query = build_user_query(["fever", "headache", "fatigue"])
results = search_disease_chroma(query, top_k=5)

for r in results:
    print(r)

{'disease': 'noninfectious gastroenteritis', 'distance': 0.38334470987319946}
{'disease': 'acute sinusitis', 'distance': 0.3981778621673584}
{'disease': 'seasonal allergies (hay fever)', 'distance': 0.4147477149963379}
{'disease': 'acute bronchitis', 'distance': 0.4190181493759155}
{'disease': 'hypertensive heart disease', 'distance': 0.43982940912246704}


In [28]:
ranked = hybrid_rank(["fever", "headache", "fatigue"])

for r in ranked[:10]:
    print(r)

{'disease': 'acute bronchitis', 'overlap': 2, 'match_ratio': 0.6666666666666666, 'chroma_distance': 0.4190181493759155}
{'disease': 'noninfectious gastroenteritis', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.38334470987319946}
{'disease': 'acute sinusitis', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.3981778621673584}
{'disease': 'seasonal allergies (hay fever)', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.4147477149963379}
{'disease': 'hypertensive heart disease', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.43982940912246704}
{'disease': 'concussion', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.44313472509384155}
{'disease': 'strep throat', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.44756388664245605}
{'disease': 'sprain or strain', 'overlap': 1, 'match_ratio': 0.3333333333333333, 'chroma_distance': 0.45756083726882935}
{'disease

In [29]:
print(generate_response(["fever", "headache", "fatigue"]))

Possible Condition: acute bronchitis
Symptom Match Count: 2
Match Ratio: 0.67

Description:
Acute bronchitis is inflammation of the bronchial tubes in the lungs, typically caused by a viral infection, resulting in cough, mucus production, chest discomfort, and low-grade fever.

Common Symptoms:
shortness of breath, headache, sharp chest pain, wheezing, fever, coughing up sputum

Medications:
nsaids, cough suppressants (e.g., dextromethorphan), expectorants (e.g., guaifenesin), bronchodilators (if wheezing), antibiotics (only if bacterial suspected)

Precautions:
avoid smoking, drink warm fluids, use cough suppressants if needed, rest and recover

Diet Suggestions:
hydration, vitamin c-rich foods (citrus fruits, bell peppers), avoid dairy if mucus increases, anti-inflammatory foods (ginger, turmeric), protein-rich foods (chicken, beans)

Workouts:
breathing exercises: aid recovery, rest: essential during coughing phase, walking: gradually reintroduce activity, avoid cold-air workouts: p

In [30]:
print(generate_response(["nausea", "vomiting", "stomach pain"]))

Possible Condition: gastrointestinal hemorrhage
Symptom Match Count: 2
Match Ratio: 0.67

Description:
Gastrointestinal hemorrhage is bleeding that occurs anywhere along the digestive tract, often presenting as vomiting blood or black, tarry stools, and can be caused by ulcers, varices, or cancer.

Common Symptoms:
dizziness, sharp abdominal pain, vomiting, nausea, diarrhea, blood in stool

Medications:
iv proton pump inhibitors (e.g., pantoprazole), endoscopic hemostasis, blood transfusion, octreotide (for variceal bleeding), antibiotics (e.g., ceftriaxone) if cirrhosis present

Precautions:
avoid nsaids, eat a soft bland diet, limit alcohol, follow up with gi specialist

Diet Suggestions:
avoid spicy and acidic foods, bland diet (bananas, rice, applesauce), hydration, iron-rich foods post bleeding (spinach, beans), avoid alcohol and nsaids

Workouts:
rest: avoid strenuous activity during active bleeding, breathing exercises: manage stress on the digestive system, gentle walking: only

In [31]:
print(generate_response(["chest pain", "shortness of breath"]))

Possible Condition: heart attack
Symptom Match Count: 1
Match Ratio: 0.50

Description:
A heart attack (myocardial infarction) occurs when blood flow to part of the heart is blocked, leading to chest pain, shortness of breath, nausea, and potentially life-threatening damage to heart muscle.

Common Symptoms:
shortness of breath, chest tightness, irregular heartbeat, nausea, sharp chest pain, arm pain

Medications:
beta-blockers (e.g., metoprolol), aspirin, nitroglycerin, ace inhibitors, thrombolytics or pci (percutaneous coronary intervention)

Precautions:
take prescribed medication, avoid stress, eat heart-healthy diet, monitor cholesterol and bp

Diet Suggestions:
low-sodium diet (vegetables, fresh fruits), omega-3 fatty acids (salmon, flaxseeds), whole grains (brown rice, oats), lean proteins (chicken, beans), limit saturated and trans fats (processed foods, fried foods)

Workouts:
cardiac rehabilitation: doctor-supervised program, walking: most recommended early-stage workout, sta

In [32]:
!pip install streamlit chromadb neo4j sentence-transformers pandas